In [1]:
# Install MediaPipe and download the hand-detector file
!pip -q install mediapipe
!wget -q -O /content/hand_landmarker.task https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task
!ls -la /content/hand_landmarker.task b

ls: cannot access 'b': No such file or directory
-rw-r--r-- 1 root root 7819105 Apr 26  2023 /content/hand_landmarker.task


Cell F1: the Flask app (page + prediction route)

In [3]:
import os, zipfile, numpy as np, tensorflow as tf
from tensorflow.keras import layers
from google.colab import files

print("Choose models.zip in the window that opens")
up = files.upload()
for z in up:
    zipfile.ZipFile(z).extractall('/content')
final = tf.keras.models.load_model('/content/HandTalkAI/models/handtalk_landmark_ann.keras')
classes_arr = np.load('/content/HandTalkAI/models/classes.npy', allow_pickle=True)

# Copy the weights into plain arrays so prediction needs only numpy
dense = [l for l in final.layers if isinstance(l, layers.Dense)]
bn = [l for l in final.layers if isinstance(l, layers.BatchNormalization)][0]
g, b, mean, var = bn.get_weights()
W = {f"d{i}": d.get_weights() for i, d in enumerate(dense)}

def predict(x):
    relu = lambda a: np.maximum(a, 0)
    a = relu(x @ W["d0"][0] + W["d0"][1])
    a = (a - mean) / np.sqrt(var + 1e-3) * g + b
    a = relu(a @ W["d1"][0] + W["d1"][1])
    a = relu(a @ W["d2"][0] + W["d2"][1])
    z = a @ W["d3"][0] + W["d3"][1]
    e = np.exp(z - z.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

# Check: this number should be very small (near 0)
t = np.random.randn(50, 63).astype("float32") * 0.5
print("Max difference:", np.abs(predict(t) - final.predict(t, verbose=0)).max())

Choose models.zip in the window that opens


Saving models.zip to models.zip
Max difference: 7.748604e-07


Cell F2: open the interface

In [4]:
import threading, base64, cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision
from flask import Flask, request, jsonify

# Two hand detectors: normal, and a lenient retry for hard photos
def mk(**kw):
    return vision.HandLandmarker.create_from_options(vision.HandLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path='/content/hand_landmarker.task'),
        running_mode=vision.RunningMode.IMAGE, num_hands=1, **kw))
det = mk(min_hand_detection_confidence=0.5)
det_low = mk(min_hand_detection_confidence=0.2, min_hand_presence_confidence=0.2)

def extract(img_bgr):
    """Photo -> 63 normalized landmark numbers (same logic as training) + preview image."""
    h, w = img_bgr.shape[:2]
    pad = int(0.35 * max(h, w))
    fill = tuple(int(c) for c in np.median(img_bgr.reshape(-1, 3), axis=0))
    padded = cv2.copyMakeBorder(img_bgr, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=fill)
    for d in (det, det_low):
        for im in (img_bgr, padded):
            rgb = np.ascontiguousarray(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
            res = d.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
            if not res.hand_landmarks: continue
            pts = np.array([[p.x, p.y, p.z] for p in res.hand_landmarks[0]], dtype=np.float32)
            prev = im.copy(); ih, iw = prev.shape[:2]
            for x, y in pts[:, :2] * [iw, ih]:
                cv2.circle(prev, (int(x), int(y)), max(3, iw // 120), (0, 255, 0), -1)
            pts = pts - pts[0]                                   # wrist = origin
            scale = np.linalg.norm(pts[:, :2], axis=1).max()
            if scale < 1e-6: return None, None
            return (pts / scale).flatten(), prev                 # divide by hand size
    return None, None

app = Flask(__name__)
lock = threading.Lock()          # one prediction at a time

PAGE = """
<!doctype html><html><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>HandTalkAI</title>
<style>
 body{font-family:Arial,sans-serif;background:#f4f6fb;margin:0;padding:20px;color:#222}
 .card{max-width:640px;margin:auto;background:#fff;border-radius:14px;padding:24px;box-shadow:0 2px 12px #0002}
 h1{margin-top:0} button,input[type=file]{margin:6px 6px 6px 0}
 button{background:#2f6fed;color:#fff;border:0;padding:10px 16px;border-radius:8px;cursor:pointer;font-size:15px}
 img,video{max-width:100%;border-radius:10px;margin-top:10px}
 #letter{font-size:64px;font-weight:bold;color:#2f6fed;margin:10px 0 0}
 .small{color:#666;font-size:13px}
</style></head><body><div class="card">
<h1>HandTalkAI</h1>
<p>Sign language (ASL alphabet) recognition from a hand photo, using MediaPipe hand landmarks and a neural network.</p>
<input type="file" id="f" accept="image/*">
<button id="cam">Use camera</button>
<div id="camBox" style="display:none">
  <video id="v" autoplay playsinline></video><br>
  <button id="snap">Capture and predict</button>
</div>
<canvas id="c" style="display:none"></canvas>
<div id="out"></div>
<p class="small">Limits: J and Z need motion; M, N, S, T look alike; trained on a small number of signers.</p>
</div>
<script>
const out=document.getElementById('out');
async function send(blob){
  out.innerHTML='Predicting...';
  const fd=new FormData(); fd.append('image',blob,'photo.jpg');
  const r=await fetch('predict',{method:'POST',body:fd}); const j=await r.json();
  if(j.error){out.innerHTML='<p style="color:#c0392b">'+j.error+'</p>';return;}
  out.innerHTML='<div id="letter">'+j.letter+'</div><div>Confidence: '+j.confidence+'%</div>'
    +'<div class="small">Top 3: '+j.top3+'</div><img src="data:image/jpeg;base64,'+j.preview+'">';
}
document.getElementById('f').onchange=e=>{if(e.target.files[0]) send(e.target.files[0]);};
document.getElementById('cam').onclick=async()=>{
  try{
    const s=await navigator.mediaDevices.getUserMedia({video:true});
    document.getElementById('v').srcObject=s;
    document.getElementById('camBox').style.display='block';
  }catch(err){out.innerHTML='Camera not available: '+err;}
};
document.getElementById('snap').onclick=()=>{
  const v=document.getElementById('v'),c=document.getElementById('c');
  c.width=v.videoWidth;c.height=v.videoHeight;c.getContext('2d').drawImage(v,0,0);
  c.toBlob(send,'image/jpeg',0.92);
};
</script></body></html>
"""

@app.route("/")
def home():
    return PAGE

@app.route("/predict", methods=["POST"])
def predict_route():
    data = request.files["image"].read()
    img = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
    if img is None:
        return jsonify(error="Could not read this image.")
    if max(img.shape[:2]) > 1280:
        s = 1280 / max(img.shape[:2]); img = cv2.resize(img, None, fx=s, fy=s)
    with lock:
        x, prev = extract(img)
    if x is None:
        return jsonify(error="No hand detected. Retake the photo with the whole hand visible and good light.")
    p = predict(x[None])[0]
    top = p.argsort()[::-1][:3]
    ok, buf = cv2.imencode(".jpg", prev)
    return jsonify(letter=str(classes_arr[top[0]]), confidence=round(float(p[top[0]]) * 100),
                   top3=", ".join(f"{classes_arr[i]} ({p[i]:.0%})" for i in top),
                   preview=base64.b64encode(buf).decode())
print("App ready")

App ready


In [5]:
from werkzeug.serving import make_server
from google.colab import output

try: server.shutdown()          # stop an older copy if this cell is run twice
except Exception: pass
server = make_server("0.0.0.0", 5000, app, threaded=True)
threading.Thread(target=server.serve_forever, daemon=True).start()

output.serve_kernel_port_as_window(5000)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>